# Ngày 18 · K-means và Gaussian Mixture (EM)

- Cài vòng Lloyd: gán về tâm gần nhất rồi dời tâm về trung bình
- Thấy hàm mục tiêu (inertia) không bao giờ tăng nhưng có thể kẹt ở cực tiểu địa phương
- Viết bước E và bước M cho hỗn hợp Gauss với ma trận hiệp phương sai đầy đủ

**Đọc:**
- ISLR (bản 2), §12.4.1 K-Means Clustering
- Ng & Ma, CS229 Lecture Notes (2023), Chương 10 Clustering and the k-means algorithm; §11.1 EM for mixture of Gaussians
- Arthur & Vassilvitskii (2007), k-means++: The Advantages of Careful Seeding, SODA
- Dempster, Laird & Rubin (1977), Maximum Likelihood from Incomplete Data via the EM Algorithm, JRSS-B 39(1), 1–38

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
from scipy.special import logsumexp
from scipy.stats import multivariate_normal
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.mixture import GaussianMixture
# Iris (Fisher, 1936), có sẵn trong scikit-learn: 150 bông, 4 đặc trưng, chuẩn hóa từng cột.
_iris = load_iris()
X_ir = (_iris.data - _iris.data.mean(0)) / _iris.data.std(0)
_init18 = X_ir[[0, 1, 2]]                   # ba bông setosa: khởi tạo cố tình tệ
_sk18 = KMeans(3, init=_init18, n_init=1, algorithm='lloyd', max_iter=300, tol=0).fit(X_ir)
_w0, _mu0, _cov0 = np.array([.5, .3, .2]), _sk18.cluster_centers_.copy(), np.array([np.eye(4)] * 3)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')        # max_iter = 1 cố ý: chỉ một vòng EM để so
    _gm18 = GaussianMixture(3, covariance_type='full', weights_init=_w0, means_init=_mu0,
                            precisions_init=np.array([np.linalg.inv(c) for c in _cov0]), max_iter=1, tol=0, reg_covar=1e-6).fit(X_ir)


## Vòng Lloyd của k-means

Gợi ý: assign: ma trận khoảng cách bình phương (n × K) bằng broadcasting, argmin theo trục 1. update: cụm rỗng giữ tâm cũ. kmeans: ghi inertia ngay sau mỗi bước gán, dừng khi tâm không đổi (np.allclose).

In [ ]:
def assign(X, C):
    raise NotImplementedError

def update(X, labels, C):
    # Trả mảng tâm mới; cụm không có điểm nào thì giữ nguyên tâm cũ.
    raise NotImplementedError

def kmeans(X, init, iters=100):
    # Trả (C, labels, history) với history là list inertia sau mỗi lần gán.
    raise NotImplementedError


In [ ]:
_report('d18-kmeans',
    _check('assign chọn tâm gần nhất', lambda: (np.array_equal(assign(X_ir, _init18 + .3), ((X_ir[:, None] - (_init18 + .3)[None]) ** 2).sum(2).argmin(1)), 'So với cách tính trực tiếp.')),
    _check('Cụm rỗng giữ nguyên tâm cũ', lambda: (np.allclose(update(np.array([[0., 0.], [2., 0.]]), np.array([0, 0]), np.array([[9., 9.], [5., 5.]])), [[1., 0.], [5., 5.]]), 'Không được sinh NaN.')),
    _check('Kết quả khớp KMeans(algorithm="lloyd") cùng khởi tạo', lambda: ((lambda r: np.allclose(r[0], _sk18.cluster_centers_, atol=1e-8) and np.array_equal(r[1], _sk18.labels_))(kmeans(X_ir, _init18)), f'Inertia tham chiếu {_sk18.inertia_:.2f}.')),
    _check('Inertia không bao giờ tăng', lambda: (np.all(np.diff(kmeans(X_ir, _init18)[2]) <= 1e-9), 'Mỗi bước gán và cập nhật chỉ làm hàm mục tiêu giảm hoặc giữ nguyên.')),
)


## Bước E và bước M của hỗn hợp Gauss

Gợi ý: Bước E: log p(x, k) = log w_k + multivariate_normal(mu_k, cov_k).logpdf(x); chuẩn hóa bằng logsumexp theo k. Bước M: N_k = Σ_i r_ik; w = N_k/n; mu_k = Σ r_ik x_i / N_k; cov_k = Σ r_ik (x_i − mu_k)(x_i − mu_k)ᵀ / N_k + reg·I.

In [ ]:
def e_step(X, w, mu, cov):
    # Trả (resp shape (n, K), log-likelihood trung bình mỗi mẫu).
    raise NotImplementedError

def m_step(X, resp, reg=1e-6):
    # Trả (w, mu, cov).
    raise NotImplementedError


In [ ]:
def _em18(n):
    w, mu, c, lls = _w0, _mu0, _cov0, []
    for _ in range(n):
        R, ll = e_step(X_ir, w, mu, c); lls.append(ll); w, mu, c = m_step(X_ir, R)
    return w, mu, c, lls
_report('d18-gmm',
    _check('Trách nhiệm mỗi hàng cộng lại bằng 1', lambda: (np.allclose(e_step(X_ir, _w0, _mu0, _cov0)[0].sum(1), 1), 'Bước E.')),
    _check('Một vòng EM khớp GaussianMixture(max_iter=1)', lambda: ((lambda r: np.allclose(r[0], _gm18.weights_) and np.allclose(r[1], _gm18.means_) and np.allclose(r[2], _gm18.covariances_))(_em18(1)), 'reg_covar = 1e-6.')),
    _check('Log-likelihood không giảm qua 40 vòng', lambda: (np.all(np.diff(_em18(40)[3]) >= -1e-9), 'Tính chất đơn điệu của EM.')),
)


## Dự án

Tải Digit Recognizer (python py30/download_data.py digits) hoặc dùng load_digits khi chưa có Kaggle. Chạy KMeans(n_clusters=10, n_init=10, random_state=0) trên ảnh, vẽ 10 tâm cụm thành ảnh, rồi tính adjusted_rand_score với nhãn thật. Ghi cặp chữ số nào hay bị gộp chung một cụm.